In [ ]:
#This step is to create a big data file with all atributes of genes or nodes

In [ ]:
library(dplyr)

In [2]:
#load data created in run09_coexp_data_filter_net_construction
#the file filtered_mm contain all genes or nodes significantly associated to a module and their p-values
nodes_mm <- read.delim("your_path/filtered_mm.tsv", sep ='\t', header=TRUE)


In [5]:
head (nodes_mm)
nrow (nodes_mm)

,geneid,module,mm,p_mm
,<chr>,<chr>,<dbl>,<dbl>
1,Llu_lncRNA_1008,MM2,0.8525117,5.913825e-10
2,Llu_lncRNA_10215,MM16,0.8073461,2.360476e-08
3,Llu_lncRNA_10404,MM2,0.8638453,1.929543e-10
4,Llu_lncRNA_10431,MM12,0.8088714,2.118204e-08
5,Llu_lncRNA_10431,MM16,0.8878188,1.247569e-11
6,Llu_lncRNA_10583,MM1,0.8553982,4.486487e-10


[1] 16810

In [4]:
nodes_mm$mm <- abs(nodes_mm$mm)

In [6]:
#Clean data to keep the most associated module to each gene 
nodes_mm_filter <- nodes_mm %>%
 group_by(geneid) %>%
  slice_max(order_by = mm, n = 1, with_ties = FALSE) %>%
ungroup()

In [ ]:
nrow (nodes_mm_filter)

In [8]:
write.table(nodes_mm_filter, "your_path/filtered_mm_final.tsv",sep = '\t',row.names = FALSE,quote = FALSE)

In [9]:
#Add type atribute
nodes_mm_filter <- nodes_mm_filter %>%
  mutate(type = ifelse(grepl("Llu_lncRNA", geneid), "lncRNA", "codingRNA"))


In [10]:
head (nodes_mm_filtrado)
nrow (nodes_mm_filtrado)

geneid,module,mm,p_mm,type
<chr>,<chr>,<dbl>,<dbl>,<chr>
Llu00002,MM4,0.8547456,4.778077e-10,codingRNA
Llu00005,MM19,0.8181077,1.076413e-08,codingRNA
Llu00010,MM4,0.8936571,5.826098e-12,codingRNA
Llu00014,MM7,0.8686117,1.168837e-10,codingRNA
Llu00025,MM3,0.8518996,6.265907e-10,codingRNA
Llu00026,MM2,0.8309153,3.943608e-09,codingRNA


[1] 12881

In [11]:
#Add differential expressed results information obtained in run07_EdgeR_DESEQ2_DEanalysis
#We performed a contrast analysis between C98 and C195 (different genotypes) and we focus on DE genes in all hpi (shared response)
DE_genotype_up_coding <- read.delim("your_path/C98vsC195_shared_response_up_DE_codingRNA.txt", sep ='\t', header=FALSE)
DE_genotype_down_coding <- read.delim("your_path/C98vsC195_shared_response_down_DE_codingRNA.txt", sep ='\t', header=FALSE)
DE_genotype_up_lncRNA <- read.delim("your_path/C98vsC195_shared_response_up_DE_lncRNA.txt", sep ='\t', header=FALSE)
DE_genotype_down_lncRNA <- read.delim("your_path/C98vsC195_shared_response_down_DE_lncRNA.txt", sep ='\t', header=FALSE)

In [ ]:
nrow (DE_genotype_up_coding)
nrow (DE_genotype_down_coding)
nrow (DE_genotype_up_lncRNA)
nrow (DE_genotype_down_lncRNA)

In [13]:
#Add DE atribute related 

DE_genotype_up_coding <- DE_genotype_up_coding %>%
  rename(geneid = V1) %>%
  mutate(DE_genotype_all = "up")

DE_genotype_up_lncRNA <- DE_genotype_up_lncRNA %>%
  rename(geneid = V1) %>%
  mutate(DE_genotype_all = "up")

DE_genotype_down_coding <- DE_genotype_down_coding %>%
  rename(geneid = V1) %>%
  mutate(DE_genotype_all = "down")

DE_genotype_down_lncRNA <- DE_genotype_down_lncRNA %>%
  rename(geneid = V1) %>%
  mutate(DE_genotype_all = "down")


In [14]:
#merge information

DE_genotype_allhpi <- bind_rows(
  DE_genotype_up_coding,
  DE_genotype_down_coding,
  DE_genotype_up_lncRNA,
  DE_genotype_down_lncRNA
)

In [15]:
#merge with nodes_mm_type information
nodes_mm_DE <- nodes_mm_filter %>%
  left_join(DE_genotype_allhpi, by = "geneid") %>%
  mutate(DE_genotype_all = ifelse(is.na(DE_genotype_all), "NA", DE_genotype_all))


In [16]:
head (nodes_mm_DE)
nrow (nodes_mm_DE)

geneid,module,mm,p_mm,type,DE_genotype_all
<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>
Llu00002,MM4,0.8547456,4.778077e-10,codingRNA,NA
Llu00005,MM19,0.8181077,1.076413e-08,codingRNA,NA
Llu00010,MM4,0.8936571,5.826098e-12,codingRNA,NA
Llu00014,MM7,0.8686117,1.168837e-10,codingRNA,NA
Llu00025,MM3,0.8518996,6.265907e-10,codingRNA,NA
Llu00026,MM2,0.8309153,3.943608e-09,codingRNA,NA


[1] 12881

In [17]:
#Add differential expressed results information obtained in run07_EdgeR_DESEQ2_DEanalysis
#We performed a contrast analysis between hours post infection and control condition for each genotype, and we focus on DE genes in all hpi (shared response) exclusive for each genotype
DE_hpi_up_coding <- read.delim("your_path/ctvshpi_shared_response_R_up_exclusive_codingRNA.txt", sep ='\t', header=FALSE)
DE_hpi_down_coding <- read.delim("your_path/ctvshpi_shared_response_R_down_exclusive_codingRNA.txt", sep ='\t', header=FALSE)
DE_hpi_up_lncRNA <- read.delim("your_path/ctvshpi_shared_response_R_up_exclusive_lncRNA.txt", sep ='\t', header=FALSE)
DE_hpi_down_lncRNA <- read.delim("your_path/ctvshpi_shared_response_R_down_exclusive_lncRNA.txt", sep ='\t', header=FALSE)

In [18]:
nrow (DE_hpi_up_coding)
nrow (DE_hpi_down_coding)
nrow (DE_hpi_up_lncRNA)
nrow (DE_hpi_down_lncRNA)

[1] 234

[1] 136

[1] 20

[1] 17

In [19]:
#Add atribute related

DE_hpi_up_coding <- DE_hpi_up_coding %>%
  rename(geneid = V1) %>%
  mutate(DE_hpi_all = "up")

DE_hpi_up_lncRNA <- DE_hpi_up_lncRNA %>%
  rename(geneid = V1) %>%
  mutate(DE_hpi_all = "up")

DE_hpi_down_coding <- DE_hpi_down_coding %>%
  rename(geneid = V1) %>%
  mutate(DE_hpi_all = "down")

DE_hpi_down_lncRNA <- DE_hpi_down_lncRNA %>%
  rename(geneid = V1) %>%
  mutate(DE_hpi_all = "down")

In [20]:
#merge
DE_hpi_all <- bind_rows(
  DE_hpi_up_coding,
  DE_hpi_down_coding,
  DE_hpi_up_lncRNA,
  DE_hpi_down_lncRNA
)

In [21]:
#merge with nodes_mm_DE
nodes_mm_DE <- nodes_mm_DE %>%
  left_join(DE_hpi_all, by = "geneid") %>%
  mutate(DE_hpi_all = ifelse(is.na(DE_hpi_all), "NA", DE_hpi_all))

In [22]:
head (nodes_mm_DE)
nrow (nodes_mm_DE)

geneid,module,mm,p_mm,type,DE_genotype_all,DE_hpi_all
<chr>,<chr>,<dbl>,<dbl>,<chr>,<chr>,<chr>
Llu00002,MM4,0.8547456,4.778077e-10,codingRNA,NA,NA
Llu00005,MM19,0.8181077,1.076413e-08,codingRNA,NA,NA
Llu00010,MM4,0.8936571,5.826098e-12,codingRNA,NA,NA
Llu00014,MM7,0.8686117,1.168837e-10,codingRNA,NA,NA
Llu00025,MM3,0.8518996,6.265907e-10,codingRNA,NA,NA
Llu00026,MM2,0.8309153,3.943608e-09,codingRNA,NA,NA


[1] 12881

In [23]:
#Save
write.table(nodes_mm_DE, "/mnt/c/Users/Usuario/Documents/Tesis_lupino/12_WGCNA_2025/4_atributos_nodos/atributos_bignet.tsv",sep = '\t',row.names = FALSE,quote = FALSE)